# SQL Database Design — Runnable Examples\n\nSQLite notebook showing keys, constraints, relationships, junction tables, joins, aggregation, and indexes.

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript('''
CREATE TABLE customers (
 customer_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, email TEXT NOT NULL UNIQUE
);
CREATE TABLE products (
 product_id INTEGER PRIMARY KEY, product_name TEXT NOT NULL UNIQUE,
 unit_price NUMERIC NOT NULL CHECK(unit_price >= 0)
);
CREATE TABLE orders (
 order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL,
 order_date TEXT NOT NULL, FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
 order_id INTEGER NOT NULL REFERENCES orders(order_id),
 product_id INTEGER NOT NULL REFERENCES products(product_id),
 quantity INTEGER NOT NULL CHECK(quantity > 0),
 unit_price NUMERIC NOT NULL CHECK(unit_price >= 0),
 PRIMARY KEY(order_id, product_id)
);
''')
conn.executemany("INSERT INTO customers VALUES (?, ?, ?)", [(1,"Asha Sharma","asha@example.com"),(2,"Ravi Patel","ravi@example.com")])
conn.executemany("INSERT INTO products VALUES (?, ?, ?)", [(10,"Notebook",50.0),(11,"Pen",10.0),(12,"Backpack",800.0)])
conn.executemany("INSERT INTO orders VALUES (?, ?, ?)", [(100,1,"2026-01-03"),(101,2,"2026-01-04")])
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", [(100,10,2,50.0),(100,11,3,10.0),(101,12,1,800.0)])
print("Schema created and sample data inserted.")


Schema created and sample data inserted.


## 1. Inspect the tables

In [2]:
for table in ["customers", "products", "orders", "order_items"]:
    print("\\n" + table)
    display(pd.read_sql_query(f"SELECT * FROM {table}", conn))


\ncustomers


,customer_id,full_name,email
0,1,Asha Sharma,asha@example.com
1,2,Ravi Patel,ravi@example.com


\nproducts


,product_id,product_name,unit_price
0,10,Notebook,50
1,11,Pen,10
2,12,Backpack,800


\norders


,order_id,customer_id,order_date
0,100,1,2026-01-03
1,101,2,2026-01-04


\norder_items


,order_id,product_id,quantity,unit_price
0,100,10,2,50
1,100,11,3,10
2,101,12,1,800


## 2. Join normalized tables for an order report

In [3]:
pd.read_sql_query('''
SELECT o.order_id, o.order_date, c.full_name, p.product_name,
       oi.quantity, oi.unit_price, oi.quantity * oi.unit_price AS line_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
ORDER BY o.order_id, p.product_id
''', conn)


,order_id,order_date,full_name,product_name,quantity,unit_price,line_total
0,100,2026-01-03,Asha Sharma,Notebook,2,50,100
1,100,2026-01-03,Asha Sharma,Pen,3,10,30
2,101,2026-01-04,Ravi Patel,Backpack,1,800,800


## 3. Aggregate order totals

In [4]:
pd.read_sql_query('''
SELECT o.order_id, c.full_name, SUM(oi.quantity * oi.unit_price) AS order_total
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.order_id, c.full_name ORDER BY o.order_id
''', conn)


,order_id,full_name,order_total
0,100,Asha Sharma,130
1,101,Ravi Patel,800


## 4. Foreign-key enforcement

In [5]:
try:
    conn.execute("INSERT INTO orders VALUES (?, ?, ?)", (999, 9999, "2026-01-05"))
    conn.commit()
except sqlite3.IntegrityError as exc:
    print("Expected integrity error:", exc)


Expected integrity error: FOREIGN KEY constraint failed


## 5. CHECK constraint enforcement

In [6]:
try:
    conn.execute("INSERT INTO products VALUES (?, ?, ?)", (99, "Invalid product", -1))
    conn.commit()
except sqlite3.IntegrityError as exc:
    print("Expected check-constraint error:", exc)


Expected check-constraint error: CHECK constraint failed: unit_price >= 0


## 6. Junction table rows

In [7]:
pd.read_sql_query("SELECT * FROM order_items ORDER BY order_id, product_id", conn)


,order_id,product_id,quantity,unit_price
0,100,10,2,50
1,100,11,3,10
2,101,12,1,800


## 7. Catalog price versus price at purchase

In [8]:
pd.read_sql_query('''
SELECT p.product_name, p.unit_price AS current_catalog_price,
       oi.order_id, oi.unit_price AS price_at_purchase
FROM products p JOIN order_items oi ON oi.product_id = p.product_id
ORDER BY p.product_id, oi.order_id
''', conn)


,product_name,current_catalog_price,order_id,price_at_purchase
0,Notebook,50,100,50
1,Pen,10,100,10
2,Backpack,800,101,800


## 8. Create an index for a common access pattern

In [9]:
conn.execute("CREATE INDEX idx_orders_customer_date ON orders(customer_id, order_date)")
pd.read_sql_query("PRAGMA index_list('orders')", conn)


,seq,name,unique,origin,partial
0,0,idx_orders_customer_date,0,c,0


## 9. Inspect table metadata

In [10]:
pd.read_sql_query("PRAGMA table_info('order_items')", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,1,None,1
1,1,product_id,INTEGER,1,None,2
2,2,quantity,INTEGER,1,None,0
3,3,unit_price,NUMERIC,1,None,0


## 10. Design review checklist\n\n- Clear table purpose and keys\n- Correct cardinality and optionality\n- Junction tables for M:N relationships\n- Constraints for business rules\n- Normalization and measured indexing\n- Migration, privacy, backup, and recovery plans